# Proyecto Final — Etapa 2: Mark 7 — HPO con Optuna

**Estrategia:**
1. **Fase búsqueda** — Optuna (TPE bayesiano) con `FULL_TRAIN=False`, 1 seed, early stop rápido
2. **Fase final** — mejores hiperparámetros + `FULL_TRAIN=True` + 3 semillas

**Espacio de búsqueda:**
| Param | Rango |
|---|---|
| `lr` | log-uniform [5e-6, 5e-5] |
| `max_len` | {128, 256} |
| `label_smooth` | uniform [0.0, 0.12] |
| `alpha` (ordinal) | uniform [0.0, 0.40] |
| `warmup_frac` | uniform [0.05, 0.20] |

**Modelo:** `PlanTL-GOB-ES/roberta-base-bne` — Apache 2.0

## 1. Instalación

In [ ]:
!pip install transformers accelerate optuna -q

## 2. Librerías

In [ ]:
import os, re, random, gc
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"PyTorch: {torch.__version__}  |  Optuna: {optuna.__version__}")

## 3. Configuración global

In [ ]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB  = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/machine-learning-project"
    OUT_DIR  = "/kaggle/working"
elif IS_COLAB:
    DATA_DIR = "/content"
    OUT_DIR  = "/content"
else:
    DATA_DIR = "../../data"
    OUT_DIR  = "../submissions"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
EVAL_PATH  = os.path.join(DATA_DIR, "eval.csv")
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME  = "PlanTL-GOB-ES/roberta-base-bne"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP     = torch.cuda.is_available()

# --- Presupuesto de búsqueda ---
N_TRIALS    = 15        # trials de Optuna; cada uno ~20-35 min en T4
TIMEOUT_SEC = 5 * 3600  # máximo 5 h para la búsqueda; deja margen para entrenamiento final

# --- Fase final ---
FINAL_SEEDS  = [42, 2024, 123]
FINAL_EPOCHS = 6

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def make_amp():
    if USE_AMP:
        try:
            return torch.amp.GradScaler("cuda"), lambda: torch.amp.autocast("cuda")
        except AttributeError:
            return torch.cuda.amp.GradScaler(), torch.cuda.amp.autocast
    return None, nullcontext

print(f"Entorno: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")
print(f"Device:  {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 4. Datos

In [ ]:
def clean(text):
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", str(text))
    return re.sub(r" {2,}", " ", text).strip()

train_df = pd.read_csv(TRAIN_PATH)
eval_df  = pd.read_csv(EVAL_PATH)
train_df["text"] = train_df["text"].map(clean)
eval_df["text"]  = eval_df["text"].map(clean)

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["decade"])
NUM_CLASSES = len(le.classes_)

# Split fijo para que todos los trials sean comparables
train_split, val_split = train_test_split(
    train_df, test_size=0.1, random_state=42, stratify=train_df["label"]
)

print(f"Train: {len(train_split)}  |  Val: {len(val_split)}  |  Eval: {len(eval_df)}  |  Clases: {NUM_CLASSES}")

## 5. Funciones de entrenamiento

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class DecadeDataset(Dataset):
    def __init__(self, texts, labels, max_len):
        self.texts   = texts.reset_index(drop=True)
        self.labels  = labels.reset_index(drop=True)
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(str(self.texts[idx]), max_length=self.max_len,
                        padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(int(self.labels[idx]), dtype=torch.long),
        }


def make_loaders(train_s, val_s, max_len, batch_size):
    tr = DataLoader(DecadeDataset(train_s["text"], train_s["label"], max_len),
                    batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=USE_AMP)
    va = DataLoader(DecadeDataset(val_s["text"],   val_s["label"],   max_len),
                    batch_size=batch_size * 2, num_workers=2, pin_memory=USE_AMP)
    return tr, va


def build_ordinal_matrix(num_classes):
    w = torch.zeros(num_classes, num_classes, device=DEVICE)
    for i in range(num_classes):
        for j in range(num_classes):
            w[i, j] = abs(i - j)
    return w / w.max()


ORDINAL_W = build_ordinal_matrix(NUM_CLASSES)


def ordinal_ce_loss(logits, labels, label_smooth, alpha):
    ce    = F.cross_entropy(logits, labels, label_smoothing=label_smooth)
    probs = F.softmax(logits, dim=-1)
    pen   = (probs * ORDINAL_W[labels]).sum(-1).mean()
    return (1 - alpha) * ce + alpha * pen


def train_epoch(model, loader, optimizer, scheduler, scaler, autocast_fn, grad_accum, label_smooth, alpha):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        with autocast_fn():
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = ordinal_ce_loss(logits, lbls, label_smooth, alpha) / grad_accum
        if scaler: scaler.scale(loss).backward()
        else:      loss.backward()
        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            if scaler: scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if scaler: scaler.step(optimizer); scaler.update()
            else:      optimizer.step()
            scheduler.step(); optimizer.zero_grad()
        total_loss += loss.item() * grad_accum
        correct    += (logits.detach().argmax(-1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / len(loader), correct / n


@torch.no_grad()
def eval_epoch(model, loader, autocast_fn):
    model.eval()
    correct, n = 0, 0
    for batch in loader:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        with autocast_fn():
            logits = model(input_ids=ids, attention_mask=mask).logits
        correct += (logits.argmax(-1) == lbls).sum().item()
        n       += len(lbls)
    return correct / n


print("Funciones listas")

## 6. Objetivo Optuna — Fase búsqueda

In [ ]:
def objective(trial):
    # --- Espacio de búsqueda ---
    lr           = trial.suggest_float("lr",           5e-6,  5e-5, log=True)
    max_len      = trial.suggest_categorical("max_len", [128, 256])
    label_smooth = trial.suggest_float("label_smooth", 0.0,   0.12)
    alpha        = trial.suggest_float("alpha",        0.0,   0.40)
    warmup_frac  = trial.suggest_float("warmup_frac",  0.05,  0.20)

    # batch ajustado según longitud para no romper memoria
    batch_size = 16 if max_len == 256 else 32
    grad_accum = 2  if max_len == 256 else 1

    scaler, autocast_fn = make_amp()
    set_seed(42)

    tr_loader, va_loader = make_loaders(train_split, val_split, max_len, batch_size)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
    ).to(DEVICE)

    no_decay = ["bias", "LayerNorm.weight"]
    opt_params = [
        {"params": [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay) and "classifier" not in n],
         "lr": lr / 10, "weight_decay": 0.01},
        {"params": [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay) and "classifier" not in n],
         "lr": lr / 10, "weight_decay": 0.0},
        {"params": [p for n, p in model.named_parameters() if "classifier" in n],
         "lr": lr, "weight_decay": 0.01},
    ]
    optimizer = AdamW(opt_params)

    max_epochs  = 12
    patience    = 2
    steps_total = (len(tr_loader) // grad_accum) * max_epochs
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer, max(1, int(warmup_frac * steps_total)), steps_total
    )

    best_val, no_improve = 0.0, 0
    for epoch in range(1, max_epochs + 1):
        train_epoch(model, tr_loader, optimizer, scheduler, scaler, autocast_fn,
                    grad_accum, label_smooth, alpha)
        val_acc = eval_epoch(model, va_loader, autocast_fn)

        trial.report(val_acc, epoch)
        if trial.should_prune():
            del model; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            raise optuna.TrialPruned()

        if val_acc > best_val:
            best_val, no_improve = val_acc, 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

        print(f"  Trial {trial.number} Epoch {epoch}  val_acc={val_acc:.4f}  best={best_val:.4f}")

    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_val


print("Objetivo definido")

## 7. Búsqueda con Optuna

In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=3)
study   = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

print(f"Iniciando búsqueda: {N_TRIALS} trials, timeout={TIMEOUT_SEC//3600}h")
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

print(f"\n{'='*60}")
print(f"Trials completados: {len(study.trials)}")
print(f"Mejor val_acc:      {study.best_value:.4f}")
print(f"Mejores params:")
for k, v in study.best_params.items():
    print(f"  {k:20s} = {v}")

## 8. Resumen de todos los trials

In [ ]:
df_trials = study.trials_dataframe().sort_values("value", ascending=False)
cols = ["number", "value"] + [c for c in df_trials.columns if c.startswith("params_")]
print(df_trials[cols].head(10).to_string(index=False))

## 9. Entrenamiento final con mejores hiperparámetros

In [ ]:
best = study.best_params
BEST_LR           = best["lr"]
BEST_MAX_LEN      = best["max_len"]
BEST_LABEL_SMOOTH = best["label_smooth"]
BEST_ALPHA        = best["alpha"]
BEST_WARMUP_FRAC  = best["warmup_frac"]
BEST_BATCH_SIZE   = 16 if BEST_MAX_LEN == 256 else 32
BEST_GRAD_ACCUM   = 2  if BEST_MAX_LEN == 256 else 1

print("Configuración final:")
print(f"  lr           = {BEST_LR:.2e}")
print(f"  max_len      = {BEST_MAX_LEN}")
print(f"  label_smooth = {BEST_LABEL_SMOOTH:.3f}")
print(f"  alpha        = {BEST_ALPHA:.3f}")
print(f"  warmup_frac  = {BEST_WARMUP_FRAC:.3f}")
print(f"  batch_size   = {BEST_BATCH_SIZE}  grad_accum={BEST_GRAD_ACCUM}")
print(f"  seeds        = {FINAL_SEEDS}  epochs={FINAL_EPOCHS}")

In [ ]:
def fit_final(seed, tag):
    set_seed(seed)
    scaler, autocast_fn = make_amp()

    tr_loader, _ = make_loaders(train_df, val_split, BEST_MAX_LEN, BEST_BATCH_SIZE)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
    ).to(DEVICE)

    no_decay = ["bias", "LayerNorm.weight"]
    opt_params = [
        {"params": [p for n, p in model.named_parameters()
                    if not any(nd in n for nd in no_decay) and "classifier" not in n],
         "lr": BEST_LR / 10, "weight_decay": 0.01},
        {"params": [p for n, p in model.named_parameters()
                    if any(nd in n for nd in no_decay) and "classifier" not in n],
         "lr": BEST_LR / 10, "weight_decay": 0.0},
        {"params": [p for n, p in model.named_parameters() if "classifier" in n],
         "lr": BEST_LR, "weight_decay": 0.01},
    ]
    optimizer = AdamW(opt_params)
    steps     = (len(tr_loader) // BEST_GRAD_ACCUM) * FINAL_EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, max(1, int(BEST_WARMUP_FRAC * steps)), steps
    )

    ckpt = os.path.join(OUT_DIR, f"ckpt_final_{tag}.pt")
    for epoch in range(1, FINAL_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(
            model, tr_loader, optimizer, scheduler, scaler, autocast_fn,
            BEST_GRAD_ACCUM, BEST_LABEL_SMOOTH, BEST_ALPHA
        )
        print(f"[{tag}] Epoch {epoch}/{FINAL_EPOCHS}  tr_acc={tr_acc:.4f}")
        torch.save(model.state_dict(), ckpt)

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model


final_models = []
for seed in FINAL_SEEDS:
    print(f"\n{'='*50}\nEntrenando seed={seed} (100% datos)\n{'='*50}")
    m = fit_final(seed, tag=f"seed{seed}")
    final_models.append(m)

print(f"\nEnsemble de {len(final_models)} modelos listo")

## 10. Inferencia con sliding window + ensemble y Envío

In [ ]:
CLS_ID = tokenizer.cls_token_id or tokenizer.bos_token_id
SEP_ID = tokenizer.sep_token_id or tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id
CHUNK  = BEST_MAX_LEN - 2
STRIDE = BEST_MAX_LEN // 2
_, autocast_fn = make_amp()


@torch.no_grad()
def get_probs(model, texts):
    all_probs = []
    model.eval()
    for text in texts:
        tids = tokenizer.encode(str(text), add_special_tokens=False)
        wins_ids, wins_mask = [], []
        pos = 0
        while pos < max(len(tids), 1):
            chunk   = tids[pos: pos + CHUNK]
            seq     = [CLS_ID] + chunk + [SEP_ID]
            pad_len = BEST_MAX_LEN - len(seq)
            wins_ids.append(seq + [PAD_ID] * pad_len)
            wins_mask.append([1] * len(seq) + [0] * pad_len)
            pos += STRIDE
            if pos >= len(tids): break
        t_ids  = torch.tensor(wins_ids,  dtype=torch.long).to(DEVICE)
        t_mask = torch.tensor(wins_mask, dtype=torch.long).to(DEVICE)
        with autocast_fn():
            logits = model(input_ids=t_ids, attention_mask=t_mask).logits
        all_probs.append(F.softmax(logits.mean(0), dim=-1).cpu())
    return torch.stack(all_probs)


print("Generando predicciones del ensemble...")
texts_eval = eval_df["text"].tolist()

ensemble_probs = sum(get_probs(m, texts_eval) for m in final_models) / len(final_models)
preds = ensemble_probs.argmax(-1).numpy()

submission = pd.DataFrame({"id": eval_df["id"], "answer": le.inverse_transform(preds)})
sub_path   = os.path.join(OUT_DIR, "submission_mark7.csv")
submission.to_csv(sub_path, index=False)
print(f"Submission guardado: {sub_path}")
print(f"Distribución:\n{submission['answer'].value_counts().sort_index()}")
submission.head()